# Oracle Tutor — Remote Model Training

Fine-tunes `all-MiniLM-L6-v2` on MTG card data and exports an ONNX artifact.

**Before running:**
1. Export the training dataset locally:
   ```bash
   cd backend
   uv run python -m ot_backend.embed.pipeline --export-dataset training-dataset.json
   ```
2. Upload `training-dataset.json` when prompted in Cell 3.
3. After training, download `onnx-model.zip` from Cell 7.
4. Locally, unzip into your `SEMANTIC_MODEL_PATH` directory, then run:
   ```bash
   uv run python -m ot_backend.embed.pipeline --reembed-only
   ```

**Runtime:** Set to `T4 GPU` (Runtime → Change runtime type).

In [ ]:
# Cell 1 — Install dependencies
import os
os.environ["WANDB_MODE"] = "disabled"

!pip install -q "sentence-transformers>=3.3.1" "optimum[onnxruntime]" accelerate datasets

In [ ]:
# Cell 2 — Verify GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3 — Upload training dataset
from google.colab import files
print('Upload your training-dataset.json file:')
uploaded = files.upload()
dataset_path = list(uploaded.keys())[0]
print(f'Loaded: {dataset_path}')

In [ ]:
# Cell 4 — Load dataset
import json
from pathlib import Path

def load_training_dataset(input_path):
    payload = json.loads(Path(input_path).read_text(encoding='utf-8'))
    version = int(payload['version'])
    if version not in (2, 3):
        raise ValueError(f'Unsupported dataset version: {version}')
    face_texts = {
        (str(r['oracle_id']), int(r['face_ix'])): str(r['text'])
        for r in payload['face_texts']
    }
    pair_ids = [
        ((str(l[0]), int(l[1])), (str(r[0]), int(r[1])))
        for l, r in payload['pair_ids']
    ]
    direct_text_pairs = [(str(a), str(b)) for a, b in payload.get('direct_text_pairs', [])]
    return face_texts, pair_ids, direct_text_pairs, payload

face_texts, pair_ids, direct_text_pairs, meta = load_training_dataset(dataset_path)
print(f'Faces: {len(face_texts):,}')
print(f'Pair IDs: {len(pair_ids):,}')
print(f'Direct text pairs: {len(direct_text_pairs):,}')
print(f'Total training examples: {len(pair_ids) + len(direct_text_pairs):,}')

In [ ]:
# Cell 5 — Training configuration (adjust as needed)
BASE_MODEL   = 'sentence-transformers/all-MiniLM-L6-v2'
EPOCHS       = 2    # 2 epochs is the sweet spot for domain adaptation at this dataset size
BATCH_SIZE   = 32   # larger than local default (8) — GPU can handle it
WARMUP_DIV  = 20
MIN_WARMUP   = 100
OUTPUT_DIR   = Path('/content/model-run')
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
# Cell 6 — Train
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses

class LazyInputExampleDataset:
    def __init__(self, pair_ids, face_texts, direct_text_pairs=None):
        self._pair_ids = pair_ids
        self._face_texts = face_texts
        self._direct = direct_text_pairs or []
        self._id_count = len(pair_ids)

    def __len__(self):
        return self._id_count + len(self._direct)

    def __getitem__(self, i):
        if i < self._id_count:
            l, r = self._pair_ids[i]
            return InputExample(texts=[self._face_texts[l], self._face_texts[r]])
        a, b = self._direct[i - self._id_count]
        return InputExample(texts=[a, b])

print(f'Loading base model: {BASE_MODEL}')
model = SentenceTransformer(BASE_MODEL)

dataset = LazyInputExampleDataset(pair_ids, face_texts, direct_text_pairs)
dataloader = DataLoader(dataset, shuffle=False, batch_size=BATCH_SIZE)
loss_fn = losses.MultipleNegativesRankingLoss(model)

total_examples = len(dataset)
warmup_steps = max(MIN_WARMUP, total_examples // WARMUP_DIV)
checkpoint_path = str(OUTPUT_DIR / 'checkpoints')

print(f'Examples: {total_examples:,}  Epochs: {EPOCHS}  Batch: {BATCH_SIZE}  Warmup: {warmup_steps}')

model.fit(
    train_objectives=[(dataloader, loss_fn)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    show_progress_bar=True,
    checkpoint_path=checkpoint_path,
)
print('Training complete.')

pytorch_path = str(OUTPUT_DIR / 'pytorch')
model.save(pytorch_path)
print(f'PyTorch model saved to {pytorch_path}')

In [ ]:
# Cell 7 — Compute embeddings (while model is in memory, no re-load needed)
import numpy as np

face_keys = sorted(face_texts.keys())
texts_to_embed = [face_texts[k] for k in face_keys]
oracle_ids_arr = np.array([k[0] for k in face_keys])
face_ixs_arr = np.array([k[1] for k in face_keys], dtype=np.int32)

print(f'Embedding {len(texts_to_embed):,} faces ...')
embeddings = model.encode(
    texts_to_embed,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

embeddings_path = OUTPUT_DIR / 'embeddings.npz'
np.savez_compressed(
    str(embeddings_path),
    oracle_ids=oracle_ids_arr,
    face_ixs=face_ixs_arr,
    embeddings=embeddings,
)
print(f'Embeddings saved. shape={embeddings.shape}  file={embeddings_path}')

In [ ]:
# Cell 10 — Package and download
import shutil

# Bundle onnx-export/ + pytorch/ + embeddings.npz into one zip.
# onnx-export/ becomes the root of SEMANTIC_MODEL_PATH when unzipped.
shutil.copytree(pytorch_path, str(onnx_path / 'pytorch'))
shutil.copy(str(embeddings_path), str(onnx_path / 'embeddings.npz'))

zip_path = '/content/onnx-model'
shutil.make_archive(zip_path, 'zip', str(onnx_path))
print(f'Created {zip_path}.zip')

files.download(f'{zip_path}.zip')
print('Download started.')

In [ ]:
## After downloading

```bash
# Unzip into your semantic model directory
unzip onnx-model.zip -d /path/to/your/SEMANTIC_MODEL_PATH

# Load pre-computed embeddings into DB (no local inference needed)
cd backend
SEMANTIC_MODEL_PATH=/path/to/your/SEMANTIC_MODEL_PATH \
  uv run python -m ot_backend.embed.pipeline \
      --load-embeddings /path/to/your/SEMANTIC_MODEL_PATH/embeddings.npz
```

## After downloading

```bash
# Unzip into your semantic model directory
unzip onnx-model.zip -d /path/to/your/SEMANTIC_MODEL_PATH

# Recompute DB embeddings using the new model
cd backend
SEMANTIC_MODEL_PATH=/path/to/your/SEMANTIC_MODEL_PATH \
  uv run python -m ot_backend.embed.pipeline --reembed-only
```